# 02. Sen4AgriNet development parcels

[Notebook 01](01_external_reference_audit.ipynb) explains why Sen4AgriNet contributes carrot and potato to development. Here we prepare declared crop-mask references, resume saved batches and collect annual Sentinel-2 imagery. The top-up targets potato training using the existing split assignments.

The outputs saved in this notebook come from an earlier run and may reflect a different crop or archive path. Use the settings below for a new run and [Notebook 08](08_dataset_summary.ipynb) for counts in the saved experiment.

## 1. Settings for this run

Review the editable settings before running the notebook from top to bottom.

| Setting | What it controls |
|---|---|
| `reference_mode` | `reuse` reads local references. `download_missing` can obtain missing reference masks. `offline` uses local references and installed dependencies and disables satellite access. |
| `reference_target_crops` | Crop requested for new reference scanning and new batches. Saved plans can resume with their original crops. |
| `reference_target_parcels_per_crop`, `reference_scan_max_patches` | Reference pool target and maximum patches to scan. The scan may stop before the target if the bounded inventory is exhausted. |
| `dataset_revision` | Release to request. `None` reuses the first pinned release. |
| `extra_batches` | Highest numbered extra batch slot to consider per source in the active round. See the rerun example below. |
| `run_satellite_collection`, `submit_only` | Enable Earth Engine work and choose whether to queue exports only or also retrieve available results. |
| `download_workers`, `intervals_per_request` | Transport concurrency and the number of intervals retrieved per request. |

### Campaigns, rounds and batches

A **campaign** holds the collection goal and its records. For example, the default training goal is 60 usable parcels and 1,200 usable pixels per crop. A **round** groups batches under a folder such as `rounds/0002`. Several notebook runs can work in that folder. The planner uses the round number in the saved campaign feedback, or round 1 if there is no feedback. This notebook resumes that round. Moving to a new round requires updated feedback.

A **batch** is one group of selected parcels and its imagery collection plan. Sen4AgriNet has a regular batch and can use numbered extra slots within the round. Each new batch can select up to 50 candidates per requested crop, or fewer if the eligible pool is small.

With `extra_batches = 25`, the planner considers extra slots `0001` through `0025`. Rerunning with 25 in the same round checks those same slots, skips completed extra batches and resumes unfinished work. Changing **25 to 26** makes slot `0026` available in that same round. The planner fills it if it reaches that slot and finds eligible, previously unselected parcels.

Archive and reference directories are resolved from project settings. An Earth Engine project is required when satellite collection is enabled.

In [9]:
# Settings for the matched development top-up.
reference_mode = "download_missing"  # reuse | download_missing | offline
run_satellite_collection = True  # offline always disables satellite calls
submit_only = False  # Queue exports. Set False to collect finished results.
extra_batches = 0  # One separately named top-up batch.
download_workers = 2
intervals_per_request = 6
reference_target_crops = ("potato",)
reference_target_parcels_per_crop = 1300  # Reference pool budget, separate from the round limit.
reference_scan_max_patches = 10013  # Explicit cap; increase deliberately to expand the pool.
dataset_revision = None  # Reuse the first pinned release.


In [10]:
from pathlib import Path
import os
import sys
repository_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), Path.cwd())
source_folder = str(repository_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)
from crop_pipeline import notebook_setup, source_workflow
from crop_pipeline.notebook_display import show_table, show_mapping
reference_options = source_workflow.reference_options(reference_mode)
notebook_setup.prepare_environment(repository_root, install_missing=reference_mode != "offline")
os.chdir(repository_root)

from crop_pipeline.sources import adaptive, annual
from crop_pipeline import progress as pipeline_progress
settings_path = Path("settings.local.json")
local_settings = notebook_setup.ensure_local_settings(settings_path)
report_folder = Path(local_settings.get("report_root", "./data/pipeline_work/inventory")).expanduser().resolve()
annual_archive_root = annual.archive_root(local_settings, report_folder, repository_root=repository_root)
run_name = adaptive.COLLECTION_NAME
candidate_parcels_per_crop = source_workflow.ROUND_PARCELS_PER_CROP
run_satellite_collection = run_satellite_collection and reference_mode != "offline"
authenticate_now = run_satellite_collection

from crop_pipeline.sources import sen4agrinet as sen4agrinet_references, bounded_references
source_name = "sen4agrinet"
inward_buffer_metres = 0  # Preserve declared crop-mask cells; no additional erosion.
dataset_folder = Path(os.environ.get("S4A_ROOT", local_settings.get("dataset_root", "./ground_truth"))).expanduser().resolve()
reference_root = Path(local_settings.get("sen4agrinet_reference_root", str(annual_archive_root / "sen4agrinet_references"))).expanduser().resolve()

# Resolve account configuration before source transfers; authentication occurs during collection.
earth_engine_project = notebook_setup.project_for_collection(local_settings, settings_path, required=run_satellite_collection)
show_mapping({
    "Requested crops": ", ".join(reference_target_crops).title(),
    "Reference mode": reference_mode,
    "Reference pool target per crop": reference_target_parcels_per_crop,
    "Patch scan limit": reference_scan_max_patches,
    "Candidates per crop per batch": candidate_parcels_per_crop,
    "Extra batch slots": extra_batches,
    "Satellite collection": "Enabled" if run_satellite_collection else "Disabled",
    "Submit only": "Yes" if submit_only else "No",
}, title="Effective collection settings")
notebook_progress = pipeline_progress.start_notebook(
    '01_sen4agrinet_preparation', source_workflow.STAGES, report_folder.parent / "progress",
    workload={"folder": str(annual_archive_root), "source": source_name},
)

Notebook environment ready: ./.venv/bin/python


Item,Value
Requested crops,Potato
Reference mode,download_missing
Reference pool target per crop,"1,300"
Patch scan limit,"10,013"
Candidates per crop per batch,50
Extra batch slots,0
Satellite collection,Enabled
Submit only,No


01_sen4agrinet_preparation: stages:   0%|          | 0/5 [00:00<?, ?stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]

## 2. Build the reference pool

Read the pinned release and reuse verified checkpoints. When missing-reference downloads are enabled, the scan reads label masks, parcel masks and coordinates through byte ranges. The report describes the scanned subset and shows how many candidate parcels are still needed.

### From Sen4AgriNet patches to field locations

A **reference patch** is a small tile from the Sen4AgriNet dataset. It can contain several fields. A **batch** is a group of selected parcels for imagery collection, which can come from several patches.

1. The notebook lists the patch files in the pinned Sen4AgriNet release, then scans them for the requested crop. For each patch it needs the crop-label mask, parcel-ID mask and map grid. Where supported, it fetches just those parts of the file by byte range and caches the reference work locally.
2. The reader checks that the masks share the same 10 m grid. Cells with the same parcel ID and target crop code are traced into a field footprint and converted to longitude and latitude (WGS 84). The source masks supply both the crop label and the field location.
3. The batch planner selects eligible fields from that reference pool and saves the plan. Earth Engine then uses the planned footprints to find Sentinel-2 imagery for the corresponding years.

The patch scan limit counts source tiles inspected. The reference pool target counts eligible parcels found inside those tiles.

In [11]:
pipeline_progress.begin_stage("Obtain references")
repository_manifest = sen4agrinet_references.discover_release(
    reference_root, enabled=reference_options["download_if_missing"], revision=dataset_revision,
)
reference_summary = bounded_references.prepare(
    repository_manifest, reference_root, dataset_folder,
    parcels_per_crop=reference_target_parcels_per_crop, seed=adaptive.DEFAULT_POLICY["seed"],
    download_enabled=reference_options["download_if_missing"], scan_enabled=True,
    target_crops=reference_target_crops, max_patches=reference_scan_max_patches,
    cached_only=reference_options["cached_only"],
)
reference_note = f"{reference_summary.get('scope', 'verified_full_release')}; {reference_summary['files_scanned']} patches; {reference_summary.get('stop_reason', 'full_inventory_reused')}"
print("Pinned release:", repository_manifest["revision"])
print("Reference coverage:", reference_note)
show_mapping(reference_summary.get("deficits", {}), title="Reference deficits", value_label="Parcels still needed")
pipeline_progress.end_stage()


01_sen4agrinet_preparation: Gathering reference patches:   0%|          | 0/5 [00:00<?, ?patch/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]

Pinned release: 4eb44157012b9ec251767721d6bae55cf0c564e2
Reference coverage: bounded_training_reference_subset; 731 patches; parcel_targets_reached


Crop,Parcels still needed
Potato,0


01_sen4agrinet_preparation: stages:  20%|██        | 1/5 [00:00<00:00, 2321.14stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]           

## 3. Check parcel eligibility

Load the verified inventory and inspect eligible parcel and pixel counts by crop. The reference pipeline checks crop codes, geometry, coordinate grids and parcel identity. Sen4AgriNet uses the declared crop-mask cells with a **0 m additional inward buffer**.

In [12]:
pipeline_progress.begin_stage("Validate references")
reference_features, inventory_provenance = annual.load_sen4agrinet_references(Path(reference_summary["field_root"]))
show_mapping(reference_summary["crop_counts"], title="Eligible reference parcels and pixels")
print("Excluded ambiguous groups:", reference_summary["excluded_parcel_groups"])
print("Edge handling: retain declared crop-mask cells (0 m additional buffer).")
pipeline_progress.end_stage()


01_sen4agrinet_preparation: Validate references:  20%|██        | 1/5 [00:00<00:00, 222.13stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]

Crop,Fragments,Parcels
Carrot,71,67
Potato,"1,340","1,327"


01_sen4agrinet_preparation: stages:  40%|████      | 2/5 [00:00<00:00, 10.52stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]              

Excluded ambiguous groups: 0
Edge handling: retain declared crop-mask cells (0 m additional buffer).


## 4. Plan or resume batches

Resume the saved plans first. The planner uses previously unselected parcels for new requests and respects locked partitions. The table shows the selected parcels and planned imagery tiles for each batch with a plan.

In [13]:
pipeline_progress.begin_stage("Prepare collection round")
adaptive_batches = adaptive.prepare_round_batches(
    "sen4agrinet", reference_features, annual_archive_root, report_folder,
    provenance={"dataset": sen4agrinet_references.DATASET_NAME,
                "dataset_revision": repository_manifest["revision"],
                "coverage": reference_summary.get("scope", "verified_full_release"),
                "reference_stop_reason": reference_summary.get("stop_reason", "full_inventory_reused"),
                "reference_scanned_files": reference_summary.get("scanned_files", []),
                "repository_files_scanned": reference_summary["files_scanned"],
                "reference_summary": str(Path(reference_summary["field_root"]).parent / "completed_inventory.json"),
                "inventories": inventory_provenance},
    inward_buffer_metres=inward_buffer_metres, parcels_per_crop=candidate_parcels_per_crop, extra_batches=extra_batches,
    candidate_crops=reference_target_crops,
    development_top_up={"potato": ("train",)},
)
annual_plan, adaptive_context = next(((plan, context) for plan, context in adaptive_batches if plan is not None), (None, adaptive_batches[-1][1]))
# A batch without a plan has no new imagery workload to display.
planned_batches = [
    {"Batch": Path(context["folder"]).name, "Selected parcels": context["candidate_parcels"],
     "Planned tiles": plan["estimate"]["tiles"], "Years": ", ".join(map(str, sorted(plan["years"])))}
    for plan, context in adaptive_batches if plan is not None
]
show_table(planned_batches, title="Parcel and imagery workload by batch")
pipeline_progress.set_workload({"folder": str(annual_archive_root), "source": source_name,
    "plans": [plan["revision"] for plan, context in adaptive_batches if plan is not None],
    "satellite_collection": run_satellite_collection})
pipeline_progress.end_stage()

01_sen4agrinet_preparation: Prepare collection round:  40%|████      | 2/5 [00:00<00:00, 10.20stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]

Development top-up: 0 candidate groups overlap saved test footprints.
Round 2: resuming saved sen4agrinet selections and original collection plans.
Round 2: up to 50 candidates per crop in each new batch. Resume/submit 1 batches.


Batch,Selected parcels,Planned tiles,Years
sen4agrinet_matched_top_up_potato_train,50,75,2019


01_sen4agrinet_preparation: stages:  60%|██████    | 3/5 [00:00<00:00,  8.79stage/s, elapsed 0m 00s; notebook ETA learning (future stages not yet timed)]                  

## 5. Collect annual imagery

With `submit_only = True`, the notebook queues the Earth Engine exports and saves the task IDs for later. Once submission finishes, you can close the laptop while Earth Engine processes the imagery and writes the results to cloud assets.

Later, rerun with `submit_only = False`. The notebook checks the saved tasks, retrieves completed results, caches the selected-pixel tables as Parquet and assembles annual TIFFs. It reuses matching verified files on later runs, so completed work can be picked up without repeating the same transfer. The laptop needs to be on for submission and for local retrieval and assembly.

Annual TIFFs store 24 half-month periods with seven measurements per period. Detailed task and transport messages go to the progress log.

In [14]:
pipeline_progress.begin_stage("Collect annual imagery")
collection_result = adaptive.collect_batches(
    adaptive_batches, earth_engine_project,
    enabled=run_satellite_collection, authenticate=authenticate_now,
    submit_only=submit_only,
    download_workers=download_workers, intervals_per_request=intervals_per_request,
)
pipeline_progress.end_stage("skipped" if not run_satellite_collection else "completed" if collection_result["status"] == "completed" else "pending")


01_sen4agrinet_preparation: stages:  80%|████████  | 4/5 [00:00<00:00, 12052.60stage/s, elapsed 6m 16s; notebook ETA learning (future stages not yet timed)]               

## 6. Review status and continue to the dataset profile

Read the batch status together with the planned parcel counts. Verified tiles describe imagery saved locally. Preprocessing determines which parcels and windows yield usable features. Use [Notebook 08](08_dataset_summary.ipynb) to compare this source with the other development and test campaigns, then use [Preprocessing 01](../preprocessing/01_imagery_quality.ipynb) to audit completed imagery.

In [15]:
pipeline_progress.begin_stage("Summary and next action")
show_mapping({"Collection status": collection_result["status"].replace("_", " ").title()},
             title="Collection result")

# Tile totals describe the batch/year ranges returned by collection.
snapshots = list(collection_result.get("progress_by_year", {}).values())
if snapshots and all("tiles" in snapshot for snapshot in snapshots):
    completed_tiles = sum(snapshot["tiles"]["complete"] for snapshot in snapshots)
    total_tiles = sum(snapshot["tiles"]["total"] for snapshot in snapshots)
    print(f"Verified local tiles · {completed_tiles:,} of {total_tiles:,}")
else:
    print("Verified local tiles · Unavailable for this run")

show_table([
    {"Batch": entry["batch"], "Status": entry["result"]["status"].replace("_", " ").title()}
    for entry in collection_result.get("batches", [])
], title="Collection status by batch")
pipeline_progress.end_stage()
pipeline_progress.finish_notebook()

01_sen4agrinet_preparation: Summary and next action:  80%|████████  | 4/5 [00:00<00:00, 545.94stage/s, elapsed 6m 16s; notebook ETA learning (future stages not yet timed)]

Item,Value
Collection status,Completed


Verified local tiles · 75 of 75


Batch,Status
sen4agrinet_matched_top_up_potato_train,Completed


01_sen4agrinet_preparation: stages: 100%|██████████| 5/5 [00:00<00:00, 427.49stage/s, elapsed 6m 16s; notebook ETA 0m 00s (future stages not yet timed)]                   

01_sen4agrinet_preparation: 6m 16s elapsed; timing report: ./data/pipeline_work/progress/01_sen4agrinet_preparation/runs/20261122T045628_b9f7edde.json


## Saved outputs

Browse the reference files, batch states and imagery through the organized collection index. The detailed progress log records the work performed by this run.

In [16]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(repository_root)
print("Organized collection outputs:", "pipeline_outputs/collection/sen4agrinet/")

Organized collection outputs: pipeline_outputs/collection/sen4agrinet/


The collection stages measure reference eligibility and imagery progress. The profile in [Notebook 08](08_dataset_summary.ipynb) reports the parcels that passed the experiment’s imagery and window requirements.